In [ ]:
from google.colab import files
print("Upload your kaggle.json:")
uploaded = files.upload()

import os, shutil
os.makedirs('/root/.config/kaggle', exist_ok=True)
shutil.move('/content/kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print("✓ Kaggle API ready")

Upload your kaggle.json:


Saving kaggle.json to kaggle.json
✓ Kaggle API ready


In [ ]:
!pip install kaggle -q

os.makedirs('/content/data', exist_ok=True)

# 140k Real and Fake Faces — ~2.5GB download
!kaggle datasets download \
    -d xhlulu/140k-real-and-fake-faces \
    -p /content/data/ \
    --unzip

print("Download complete! Checking structure...")
!find /content/data -type d

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:43<00:00, 92.8MB/s]

Download complete! Checking structure...
/content/data
/content/data/real_vs_fake
/content/data/real_vs_fake/real-vs-fake
/content/data/real_vs_fake/real-vs-fake/train
/content/data/real_vs_fake/real-vs-fake/train/real
/content/data/real_vs_fake/real-vs-fake/train/fake
/content/data/real_vs_fake/real-vs-fake/test
/content/data/real_vs_fake/real-vs-fake/test/real
/content/data/real_vs_fake/real-vs-fake/test/fake
/content/data/real_vs_fake/real-vs-fake/valid
/content/data/real_vs_fake/real-vs-fake/valid/real
/content/data/real_vs_fake/real-vs-fake/valid/fake


In [ ]:
# Check exact folder names and file counts
import os
for root, dirs, files in os.walk('/content/data'):
    if files:
        print(f"{root}  →  {len(files)} files  (e.g. {files[0]})")

/content/data  →  3 files  (e.g. valid.csv)
/content/data/real_vs_fake/real-vs-fake/train/real  →  50000 files  (e.g. 59913.jpg)
/content/data/real_vs_fake/real-vs-fake/train/fake  →  50000 files  (e.g. 2Q5IJXUFV6.jpg)
/content/data/real_vs_fake/real-vs-fake/test/real  →  10000 files  (e.g. 45772.jpg)
/content/data/real_vs_fake/real-vs-fake/test/fake  →  10000 files  (e.g. K23SYG1FLY.jpg)
/content/data/real_vs_fake/real-vs-fake/valid/real  →  10000 files  (e.g. 58824.jpg)
/content/data/real_vs_fake/real-vs-fake/valid/fake  →  10000 files  (e.g. NT5PUVREA1.jpg)


In [ ]:
import numpy as np
import cv2, os, gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TF: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
IMG_SIZE     = 128
N_PER_CLASS  = 10000   # 10k real + 10k fake = 20k total

# ⚠ Update these if Cell 3 shows different folder names
REAL_DIR = '/content/data/real_vs_fake/real-vs-fake/train/real'
FAKE_DIR = '/content/data/real_vs_fake/real-vs-fake/train/fake'

def load_folder(folder, label, limit):
    imgs, lbls = [], []
    valid = ('.jpg', '.jpeg', '.png')
    flist = [f for f in os.listdir(folder)
             if f.lower().endswith(valid)][:limit]
    for fname in flist:
        img = cv2.imread(os.path.join(folder, fname))
        if img is None: continue
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        imgs.append(img / 255.0)
        lbls.append(label)
    return imgs, lbls

print("Loading REAL images...")
real_imgs, real_lbls = load_folder(REAL_DIR, 0, N_PER_CLASS)
print(f"Loaded {len(real_imgs)} real images")

print("Loading FAKE images...")
fake_imgs, fake_lbls = load_folder(FAKE_DIR, 1, N_PER_CLASS)
print(f"Loaded {len(fake_imgs)} fake images")

X = np.array(real_imgs + fake_imgs, dtype=np.float32)
y = np.array(real_lbls + fake_lbls, dtype=np.float32)

# Shuffle
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

print(f"\nX shape : {X.shape}")
print(f"REAL    : {(y==0).sum()}")
print(f"FAKE    : {(y==1).sum()}")

Loading REAL images...
Loaded 10000 real images
Loading FAKE images...
Loaded 10000 fake images


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Top: REAL    Bottom: FAKE', fontsize=13)
real_idx = np.where(y == 0)[0][:5]
fake_idx = np.where(y == 1)[0][:5]
for i in range(5):
    axes[0,i].imshow(X[real_idx[i]]); axes[0,i].axis('off')
    axes[1,i].imshow(X[fake_idx[i]]); axes[1,i].axis('off')
plt.tight_layout(); plt.show()